In [27]:
import numpy as np 
import sys
import time
import csv
import os
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
import torch
from pathlib import Path
import hydra
from omegaconf import OmegaConf
from datetime import datetime

module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs, write_fvecs
from src.utils import append_or_create_csv


# Imports experiments (necessary to register experiments)
from lib.Qinco.qinco.qinco_tasks import QincoConvertTask, QincoEvalTask, QincoTrainTask
from lib.Qinco.qinco.search.search_tasks import (
    BuildIndexTask,
    EncodeDBTask,
    IVFTrainTask,
    SearchTask,
    TrainPairwiseDecoderTask,
)
# from lib.Qinco

In [28]:
root = 'data'
data_fp = f'/{root}/cpanourg/2-hdvc/data/' 
temp_fp = f'/{root}/cpanourg/2-hdvc/temp/' 

dataset_name = 'deep'

if dataset_name == 'gist':
    db = np.array(read_fvecs(f'{data_fp}gist/gist_base.fvecs')).astype(np.float32)
    qr = np.array(read_fvecs(f'{data_fp}gist/gist_query.fvecs')).astype(np.float32)
    
elif dataset_name == 'deep':
    db = np.fromfile(f'{data_fp}deep1b/dataset/deep1b-96-1m.bin', dtype=np.float32).reshape(1_000_000, -1)
    qr = np.fromfile(f'{data_fp}deep1b/queries/queries-hard10p-deep1b-len96-1000.bin', np.float32).reshape(1000, -1)
    
    db_fp = f'{temp_fp}db_set_{dataset_name}.fvecs'
    qr_fp = f'{temp_fp}qr_set_{dataset_name}.fvecs'
    
    write_fvecs(db_fp, db)
    write_fvecs(qr_fp, qr)


nb = db.shape[0]  # Number of database vectors
nq = qr.shape[0]  # Number of query vectors
dim = db.shape[1]  # Dimensionality of the vectors

print(f"Database shape: {db.shape}")
print(f"Query shape: {qr.shape}")
print(f"Dimension: {dim}")

dim = db.shape[1]
nbits = dim 


Writing File - /data/cpanourg/2-hdvc/temp/db_set_deep.fvecs:(1000000, 96)
Writing File - /data/cpanourg/2-hdvc/temp/qr_set_deep.fvecs:(1000, 96)
Database shape: (1000000, 96)
Query shape: (1000, 96)
Dimension: 96


In [29]:
# sampling for training 
sampling = 'random'
train_ratio = 0.001 
val_ratio = 0.2

n_samples = int(train_ratio * db.shape[0])
n_val_samples = int(n_samples * val_ratio)

train_set_fp = f'{temp_fp}{dataset_name}_train_ratio{train_ratio}.fvecs'
train_set = db[np.random.choice(db.shape[0], size=n_samples, replace=False)]

print(f"Train set size: {n_samples}")
print(f'Validation set size: {n_val_samples}')

write_fvecs(train_set_fp, train_set)

Train set size: 1000
Validation set size: 200
Writing File - /data/cpanourg/2-hdvc/temp/deep_train_ratio0.001.fvecs:(1000, 96)


In [ ]:

cfg = OmegaConf.load("/home/cpanourg/projects/2-hdvc/lib/Qinco/config/qinco_cfg.yaml")


EXPERIMENTS = {
    "train": QincoTrainTask,
    "eval_valset": QincoTrainTask,
    "eval": QincoEvalTask,
    "eval_time": QincoEvalTask,
    "convert": QincoConvertTask,
    "ivf_centroids": IVFTrainTask,
    "encode": EncodeDBTask,
    "build_index": BuildIndexTask,
    "train_pairwise_decoder": TrainPairwiseDecoderTask,
    "search": SearchTask,
}


# Get current datetime
now = datetime.now()

# Format as string: year_month_day_hour_minute_second
datetime_str = now.strftime("%Y_%m_%d_%H_%M_%S")

print(datetime_str)


cfg.task = 'train'
cfg.output = f'/{root}/cpanourg/2-hdvc/results/qinco2/qinco_weights_{datetime_str}.pt'

cfg.L = 16 # num of resblocks in each step
cfg.dh = 384

# ----------------------------------------------------------------
# these values can be set as 0 to disable these components 
cfg.de = 384 # embedding dimension (if 0 dimension is the same as data)
cfg.A = 16 # num of fast pre-selected candidates (if 0 no beam search)
cfg.B = 32 # size of beam search (if 0 then no pre-selected candidates) 
# ----------------------------------------------------------------

cfg.M = 8 # number of qinco steps 
cfg.K = 256 # codebook size 
cfg.ivf_K = 1048576

cfg.epochs = 10 # note that it runs along with the patience (=10 by default)

cfg.db = db_fp
cfg.trainset = train_set_fp 
cfg.ds.valset = n_val_samples

print(f"Checking epochs setting: {cfg.epochs}")  # Add this to verify the value
expe = EXPERIMENTS[cfg.task](cfg)


2025_10_26_16_34_11
Checking epochs setting: 10


In [6]:


expe.accelerator.print(f"====================== RUNNING TASK {cfg.task}")
expe.run()
expe.accelerator.print("Task done")
expe.accelerator.end_training()  # Destroy process group


[T_total=00:00:00 | T_train=00:00:00 | T_inference=00:00:00] inference on validation split 1 / 1 [[MSE=62.3505]]
[T_total=00:00:00 | T_train=00:00:00 | T_epoch=00:00:00] train 1 / 1 (step 0) lr=0.000266667 loss=8.11512 (avg=8.11512) [[all losses: loss_substep=3.7687 ; mse_loss=4.34642]]
[T_total=00:00:01 | T_train=00:00:00 | T_inference=00:00:00] inference on validation split 1 / 1 [[MSE=260.439]]
[T_total=00:00:02 | T_train=00:00:01 | T_epoch=00:00:00] train 1 / 1 (step 1) lr=0.000533333 loss=10.0614 (avg=12.0078) [[all losses: loss_substep=4.57052 ; mse_loss=5.49093 ; total_loss=10.0614]]
[T_total=00:00:02 | T_train=00:00:01 | T_inference=00:00:00] inference on validation split 1 / 1 [[MSE=82.3003]]
[T_total=00:00:03 | T_train=00:00:01 | T_epoch=00:00:00] train 1 / 1 (step 2) lr=0.0008 loss=9.5226 (avg=8.44492) [[all losses: loss_substep=4.34839 ; mse_loss=5.17422 ; total_loss=9.5226]]
[T_total=00:00:03 | T_train=00:00:01 | T_inference=00:00:00] inference on validation split 1 / 1 [[

In [35]:
expe.qinco_model.eval()

QINCo(
  (steps): ModuleList(
    (0): QINCoStep(
      (codebook): Embedding(256, 96)
    )
    (1-7): 7 x QINCoStep(
      (substep): QincoSubstep(
        (codebook): Embedding(256, 96)
      )
      (codebook): Embedding(256, 96)
      (concat): RecursiveScriptModule(
        original_name=QConcat
        (mlp): RecursiveScriptModule(original_name=Linear)
      )
      (residual_blocks): RecursiveScriptModule(
        original_name=Sequential
        (0): RecursiveScriptModule(
          original_name=QBlockFFN
          (up_proj): RecursiveScriptModule(original_name=Linear)
          (act): RecursiveScriptModule(original_name=ReLU)
          (down_proj): RecursiveScriptModule(original_name=Linear)
        )
        (1): RecursiveScriptModule(
          original_name=QBlockFFN
          (up_proj): RecursiveScriptModule(original_name=Linear)
          (act): RecursiveScriptModule(original_name=ReLU)
          (down_proj): RecursiveScriptModule(original_name=Linear)
        )
       

In [13]:
db_torch = torch.tensor(db).to('cuda')
qr_torch = torch.tensor(qr).to('cuda')

In [ ]:
with torch.no_grad():
    db_codes = expe.qinco_model.encode(db_torch)
    qr_codes = expe.qinco_model.encode(qr_torch)


In [ ]:
with torch.no_grad():
    db_rec = expe.qinco_model.decode(db_codes)
    qr_rec = expe.qinco_model.decode(qr_codes)


In [ ]:
t0 = time.time()
dists = torch.cdist(qr_rec, db_rec, p=2)  # Euclidean distances
print("Distance matrix shape:", dists.shape)

dists_time = time.time() - t0
print("Computed in", dists_time, "seconds")


In [ ]:
dists

In [12]:
db_enc = expe.qinco_model(db_torch)
qr_enc = expe.qinco_model(qr_torch)


KeyboardInterrupt: 

In [14]:
qr_enc = expe.qinco_model(qr_torch)


In [15]:
qr_enc

(tensor([[ 19,  12,  12,  ...,  19,  19,  12],
         [ 26,  26,  26,  ...,  26,  26,  26],
         [ 90,  90,  90,  ...,  90,  90,  90],
         ...,
         [122, 122, 122,  ..., 122, 122, 122],
         [244, 244, 244,  ..., 244, 244, 244],
         [ 11,  11,  11,  ...,  11,  11,  11]], device='cuda:0'),
 tensor([[-0.0609, -0.3986, -0.4223,  ...,  0.2942,  0.6652, -0.1498],
         [ 0.1637,  0.6298, -0.1336,  ..., -0.7374, -0.0675,  0.3175],
         [ 0.1637,  0.6298, -0.1336,  ..., -0.7374, -0.0675,  0.3175],
         ...,
         [-0.0609, -0.3986, -0.4223,  ...,  0.2942,  0.6652, -0.1498],
         [-0.0609, -0.3986, -0.4223,  ...,  0.2942,  0.6652, -0.1498],
         [ 0.1637,  0.6298, -0.1336,  ..., -0.7374, -0.0675,  0.3175]],
        device='cuda:0', grad_fn=<AddBackward0>),
 {'mse_loss': tensor(2.3966, device='cuda:0', grad_fn=<AddBackward0>),
  'loss_substep': tensor(2.1143, device='cuda:0', grad_fn=<AddBackward0>)})

In [24]:
qr_enc[2]

{'mse_loss': tensor(2.3966, device='cuda:0', grad_fn=<AddBackward0>),
 'loss_substep': tensor(2.1143, device='cuda:0', grad_fn=<AddBackward0>)}

In [20]:
qr

array([[-0.27883, -0.47586,  0.10802, ..., -0.2837 ,  0.04609,  0.01349],
       [-0.14114, -0.37729, -0.23598, ...,  0.07962,  0.22013,  0.33016],
       [ 0.26849,  0.17537, -0.27299, ...,  0.3755 , -0.09167, -0.36357],
       ...,
       [-0.16046, -0.51814, -0.55249, ...,  0.45274,  0.312  , -0.78973],
       [ 0.31378, -0.04148, -0.22728, ..., -0.09066, -0.05866,  0.20733],
       [-0.19361, -0.09325,  0.69261, ..., -0.38795,  0.25748, -0.3097 ]],
      dtype=float32)

In [ ]:
expe.